# Task 4: Forecasting Model Development

## Objective

The objective of this task is to develop forecasting models for Ethiopia's financial inclusion indicators using the enriched dataset prepared in previous tasks. This notebook prepares the forecasting dataset, trains baseline and statistical forecasting models, evaluates model performance, and generates future forecasts that will be used in the dashboard developed in Task 5.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

from statsmodels.tsa.stattools import adfuller

plt.style.use("ggplot")

In [ ]:
project_root = Path.cwd()

processed_path = project_root / "data" / "processed"

figures_path = project_root / "figures"

processed_path.mkdir(
    parents=True,
    exist_ok=True
)

figures_path.mkdir(
    parents=True,
    exist_ok=True
)

print(processed_path)

In [ ]:
forecast_df = pd.read_csv(
    processed_path / "forecast_dataset.csv"
)

forecast_df.head()

In [ ]:
print("Shape:", forecast_df.shape)

forecast_df.info()

forecast_df.describe(include="all")

In [ ]:
missing = (
    forecast_df
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

missing[missing > 0]

In [ ]:
forecast_df["analysis_year"] = (
    pd.to_numeric(
        forecast_df["analysis_year"],
        errors="coerce"
    )
)

forecast_df = forecast_df.sort_values(
    "analysis_year"
)

forecast_df.head()

In [ ]:
account_df = forecast_df[
    forecast_df["indicator_code"] == "ACC_OWNERSHIP"
].copy()

account_df = account_df.sort_values(
    "analysis_year"
)

account_df[
    [
        "analysis_year",
        "value_numeric"
    ]
]

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    account_df["analysis_year"],
    account_df["value_numeric"],
    marker="o",
    linewidth=2
)

plt.title(
    "Historical Account Ownership"
)

plt.xlabel("Year")

plt.ylabel("Percent")

plt.grid(True)

plt.tight_layout()

plt.savefig(
    figures_path /
    "historical_account_ownership.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
train_size = int(
    len(account_df) * 0.8
)

train = account_df.iloc[:train_size]

test = account_df.iloc[train_size:]

print("Training observations:", len(train))

print("Testing observations:", len(test))

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    train["analysis_year"],
    train["value_numeric"],
    marker="o",
    label="Training"
)

plt.plot(
    test["analysis_year"],
    test["value_numeric"],
    marker="o",
    label="Testing"
)

plt.title(
    "Training and Testing Split"
)

plt.xlabel("Year")

plt.ylabel("Account Ownership")

plt.legend()

plt.tight_layout()

plt.savefig(
    figures_path /
    "train_test_split.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
adf_result = adfuller(
    account_df["value_numeric"]
)

print("ADF Statistic:", adf_result[0])

print("p-value:", adf_result[1])

if adf_result[1] < 0.05:
    print("Series is stationary.")
else:
    print("Series is not stationary.")

In [ ]:
print("Forecast dataset loaded:", not forecast_df.empty)

print("Training rows:", len(train))

print("Testing rows:", len(test))

print(
    "Historical years:",
    account_df["analysis_year"].min(),
    "-",
    account_df["analysis_year"].max()
)